# 06: TFPnP Paper Analysis

This notebook explores Wei et al. (2022) *"TFPnP: Tuning-free Plug-and-Play Proximal Algorithm with Applications to Inverse Imaging Problems"*, mapping every equation and algorithm step to
concrete Python code in `ct_tfpnp/` and recording every design decision shaping our implementation.

## 1. Setup

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from scipy.ndimage import gaussian_filter
from copy import deepcopy

import ct_tfpnp
from LION.CTtools.ct_utils import make_operator
from LION.CTtools.ct_geometry import Geometry

device = torch.device("cuda")

print(f"ct_tfpnp : {ct_tfpnp.__version__}")
print(f"Device   : {device}")
print(f"GPU      : {torch.cuda.get_device_name()}")

# set consistent plotting style for all notebook cells
plt.rcParams.update({
    "figure.dpi": 150,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "image.cmap": "gray",
    "image.interpolation": "nearest",
})

# create output directory for figures
output_dir = Path("../figures/06")
output_dir.mkdir(parents=True, exist_ok=True)

ct_tfpnp : 0.1.dev5+gcb41b39e8.d20260612
Device   : cuda
GPU      : NVIDIA A100-SXM4-80GB


## 2. Paper Overview

PnP-ADMM produces high-quality reconstructions, but only if the right denoising strength $\sigma_k$ and penalty $\mu_k$ are chosen at every iteration, for every image. These parameters are image-dependent and iteration-dependent, making manual tuning impractical. 

TFPnP's answer is to frame the parameter selection as a Markov Decision Process and learn a policy $\pi$ that, given the current ADMM state, outputs the optimal $\sigma_k$, $\mu_k$, and a termination decision automatically, per image, per iteration.

The policy is trained by reinforcement learning, where the reward at each step is the PSNR increment produced by the chosen parameters, minus a small penalty $\eta$ for not stopping, encouraging both quality improvement and computational efficiency.

## 3. The MDP Formulation

TFPnP frames PnP-ADMM parameter selection as a Markov Decision Process $(\mathcal{S}, \mathcal{A}, p, r)$:

### State space $\mathcal{S}$

$$s_t = (x^k,\; z^k,\; u^k,\; \sigma_{\text{noise}},\; k/k_{\max}) \in \mathbb{R}^{5 \times H \times W}$$

The state is the full ADMM triple $(x, z, u)$ augmented with two auxiliary scalar
channels broadcast to spatial maps:
- $\sigma_{\text{noise}}$ — measurement noise level (tells the policy how noisy the input is)
- $k/k_{\max}$ — normalised iteration count (tells the policy how far along it is)

### Action space $\mathcal{A}$

The action is decomposed: $a = (a_1, a_2)$

- $a_1 \in \{0, 1\}$ — discrete termination decision (0=continue, 1=stop)
- $a_2 = (\sigma_0, \ldots, \sigma_{m-1}, \mu_0, \ldots, \mu_{m-1})$ — continuous parameters
  for the next $m=5$ ADMM iterations

This two-part decomposition drives two separate sub-policies:
- $\pi_1$ — stochastic*(samples $a_1$ from categorical), trained model-free
- $\pi_2$ — deterministic (outputs $a_2$ directly), trained model-based via backprop

### Transition function $p$

$$s_{t+1} = p(s_t, a_t)$$

Running $m=5$ ADMM iterations with the chosen $(\sigma, \mu)$ sequence:
$$x^{k+1} = \mathcal{H}_{\sigma_k}(z^k - u^k), \quad
  z^{k+1} = \text{Prox}_{\frac{1}{\mu_k}D}(x^{k+1} + u^k), \quad
  u^{k+1} = u^k + x^{k+1} - z^{k+1}$$

### Reward function $r$ (eq. 14)

$$r(s_t, a_t) = \underbrace{[\zeta(p(s_t, a_t)) - \zeta(s_t)]}_{\text{PSNR increment}} - \underbrace{\eta}_{\text{continuation penalty}}$$

where $\zeta(s)$ is the PSNR of $x$ in state $s$, and $\eta = 0.05$.

The $\eta$ term is key: if the PSNR gain from continuing is less than $\eta$, the policy
should prefer stopping. This is how TFPnP learns early stopping automatically.

In [ ]:
# implement reward function exactly as in the paper
def compute_reward(x_new, x_old, x_gt, eta=0.05, data_range=None):
    """
    r(s_t, a_t) = [PSNR(x_new, x_gt) - PSNR(x_old, x_gt)] - eta

    Args:
        x_new     : reconstruction after taking action (numpy array)
        x_old     : reconstruction before taking action
        x_gt      : ground truth image
        eta       : continuation penalty (paper uses 0.05)
        data_range: pixel value range for PSNR. If None, uses x_gt.max() —
                    the correct convention for LION-native µ images where
                    image values run [0, ~2.45] rather than [0, 1].

    Returns:
        reward: scalar. Positive = improvement exceeded eta; negative = stop encouraged.
    """
    if data_range is None:
        data_range = float(np.asarray(x_gt).max())

    # compute PSNR in dB
    def psnr(a, b):
        mse = np.mean((a - b) ** 2)
        return 10 * np.log10(data_range ** 2 / mse) if mse > 0 else float("inf")

    psnr_new = psnr(x_new, x_gt)
    psnr_old = psnr(x_old, x_gt)

    # return reward
    return (psnr_new - psnr_old) - eta

## 4. Algorithm 1

### Algorithm 1 (from the paper)

```
Input: Image dataset D, degradation operator g(·), learning rates l_θ, l_φ, weight β
1: Initialise network parameters θ, φ, φ̂ and state buffer B
2: for each training iteration do
3:     sample initial state s₀ from D via g(·)
4:     for environment step t ∈ [0, N) do
5:         aₜ ~ πθ(aₜ|sₜ)
6:         sₜ₊₁ ~ p(sₜ₊₁|sₜ, aₜ)
7:         B ← B ∪ {sₜ₊₁}
8:         break if the boolean outcome of aₜ equals to 1
9:     end for
10:    for each gradient step do
11:        sample states from the state buffer B
12:        θ₁ ← θ₁ + l_θ ∇θ₁ J(πθ)
13:        θ₂ ← θ₂ + l_θ ∇θ₂ J(πθ)
14:        φ ← φ − l_φ ∇φ L_φ
15:        φ̂ ← β φ + (1−β) φ̂
16:    end for
17: end for
Output: Learned policy network πθ
```

## 5. The Critic Loss (Equation 15)

The value network $V^\pi_\phi(s)$ is trained by minimising the TD error, which is the gap between what the critic predicts before acting, $V^\pi_\phi(s)$, and the better-informed estimate of the same quantity formed after taking one real ADMM step and observing the actual reward, $r(s,a) + \gamma V^\pi_{\hat{\phi}}(p(s,a))$. Minimising its square pulls the critic towards consistency with the rewards it actually receives.

$$\mathcal{L}_\phi = \mathbb{E}_{s \sim B,\, a \sim \pi_\theta(s)} \left[ \frac{1}{2} \left(r(s, a) + \gamma V^\pi_{\hat{\phi}}(p(s, a)) - V^\pi_\phi(s)\right)^2 \right]$$

where:
- $B$ is the replay buffer storing past ADMM states
- $r(s, a)$ is the PSNR-increment reward
- $\gamma = 0.99$ is the discount factor
- $\hat{\phi}$ is the target critic
- $p(s, a)$ is the next state after applying the ADMM step

In [ ]:
def critic_loss_fn(
    value_net,         # current critic
    target_value_net,  # target critic
    states,            # sampled from replay buffer
    rewards,           # r(s, a)
    next_states,       # p(s, a)
    gamma=0.99,
):
    """
    Critic loss from equation (15) of Wei et al.
    
    L_phi = E[ (1/2) * (r + gamma * V_hat(s') - V(s))^2 ]
    
    The target is computed with stop-gradient on target_value_net.
    """
    # current value estimate V_phi(s)
    v_current = value_net(*states)         
    
    # next value estimate 
    with torch.no_grad():
        v_next = target_value_net(*next_states)  
    
    # compute TC target
    td_target = rewards.unsqueeze(1) + gamma * v_next  
    
    # compute TD error loss
    loss = 0.5 * F.mse_loss(v_current, td_target)
    
    return loss

## 6. The Two Policy Gradient Updates

### π₁: Discrete termination (REINFORCE, model-free) — eq. 16

$$\nabla_{\theta_1} J(\pi_\theta) = \mathbb{E}_{s \sim B,\, a \sim \pi_\theta(s)} \left[ \nabla_{\theta_1} \log \pi_1(a_1|s) \cdot A^\pi(s, a) \right]$$

where $A^\pi(s, a) = Q^\pi(s, a) - V^\pi(s)$ is the advantage function.

Since $a_1$ is discrete (stop/continue), we cannot backprop through it directly. REINFORCE provides an unbiased Monte Carlo gradient estimate using the log-probability of the sampled action, weighted by the advantage.

### π₂: Continuous parameters (DDPG-style, model-based) — eq. 17

$$\nabla_{\theta_2} J(\pi_\theta) = \mathbb{E}_{s \sim B}\left[ \nabla_{\theta_2} V^\pi_\phi(p(s, \pi_2(s))) \right]$$

Since $a_2 = \pi_2(s)$ is continuous and the ADMM environment is differentiable, we can backpropagate the value gradient directly through:
$$\theta_2 \leftarrow \theta_2 + l_\theta \nabla_{\theta_2} V_\phi(\text{ADMM}(s, \sigma, \mu))$$

In [ ]:
def policy_loss_discrete(
    policy_net,
    value_net,
    states,
    actions_a1,       # sampled discrete actions
    rewards,
    next_states,
    gamma=0.99,
):
    """
    REINFORCE gradient for π₁ (discrete termination) — equation (16).
    """
    # define stop action logits 
    stop_logits, _, _ = policy_net(*states)

    # compute log-prob of the sampled action a₁ under π₁
    log_probs = F.log_softmax(stop_logits, dim=-1)             
    log_prob_a1 = log_probs.gather(1, actions_a1.unsqueeze(1)).squeeze(1) 
    
    # compute advantage
    with torch.no_grad():
        v_s  = value_net(*states).squeeze(1)         
        v_s_next = value_net(*next_states).squeeze(1)
    advantage = rewards + gamma * v_s_next - v_s    
    
    # compute REINFORCE loss
    loss = -(log_prob_a1 * advantage.detach()).mean()

    return loss


def policy_loss_continuous(
    policy_net,
    value_net,
    admm_step_fn,      # differentiable ADMM step
    states,
    sinograms,
):
    """
    DDPG-style gradient for π₂ (continuous σ, μ) — equation (17).
    """
    # unpack states
    x, z, u, noise_level, iter_frac = states
    
    # sample actions from π₂
    _, sigma_seq, mu_seq = policy_net(*states)
    
    # run m ADMM steps with chosen parameters
    x_new, z_new, u_new = x.clone(), z.clone(), u.clone()
    for step_i in range(sigma_seq.shape[1]):
        sigma_i = sigma_seq[:, step_i]
        mu_i    = mu_seq[:, step_i]
        x_new, z_new, u_new = admm_step_fn(x_new, z_new, u_new, sinograms, sigma_i, mu_i)
    
    # compute value of next state 
    n_iter = iter_frac + 1.0 / 6.0 
    next_state = (x_new, z_new, u_new, noise_level, n_iter.clamp(0, 1))
    v_next = value_net(*next_state) 
    
    # compute loss as negative val of next state
    loss = -v_next.mean()
    
    return loss

## 7. Network Architecture

### Policy network (`ResNetActor_ADMM`)

```
Input: (x, z, u, noise_map, iter_map)   → (B, 5, H, W)
  │
  ├─ conv1:   5→64,  7×7, stride 2, padding 1        →  H/2
  ├─ bn1 + relu
  ├─ maxpool: 3×3, stride 2                          →  H/4
  ├─ layer1:  2× [3×3, 64  | 3×3, 64 ] residual      →  H/4
  ├─ layer2:  2× [3×3, 128 | 3×3, 128] residual      →  H/8
  ├─ layer3:  2× [3×3, 256 | 3×3, 256] residual      →  H/16
  ├─ layer4:  2× [3×3, 512 | 3×3, 512] residual      →  H/32
  ├─ avgpool: AdaptiveAvgPool2d(1)                   →  512-dim feature vector
  │
  ├─ Termination head (π₁): Linear(512, 2)  → softmax over (continue, stop)
  └─ Parameter head   (π₂): Linear(512, 10) → sigmoid → (σ[5], µ[5]),
                                              rescaled to the configured ranges
```

Each residual block is two 3×3 convolutions with batch normalisation, ReLU, and a skip connection.

### Value network (`ResNet_wobn`)

**This is a deviation from the paper and is not Table 1's feature extractor.**

Wei et al. specify the same ResNet-18 extractor as the policy, with batch normalisation replaced by weight normalisation and ReLU replaced by TReLU, ending in `Linear(512, 1)`.

Ours is a smaller network with no normalisation and no learned activation.

```
Input: (x, z, u, noise_map, iter_map)   → (B, 5, H, W)
  │
  ├─ head: Conv2d(5 → 64, 3×3, padding 1)            →  H×W
  ├─ body: 8× ResBlock_wobn(64)                      →  H×W
  │          conv 3×3 → ReLU → conv 3×3, identity skip
  │          no normalisation, constant width, stride 1
  ├─ pool: AdaptiveAvgPool2d(1)                      →  64-dim feature vector
  └─ tail: Linear(64, 1)                             →  scalar V^π(s)
```

Batch normalisation is omitted because its running statistics shift as the policy changes during training, which destabilises value estimation. That much follows the paper's reasoning. The replacement, however, is a plain BN-free residual stack rather than the paper's weight-normalised TReLU network, and the body never downsamples, so all eight blocks run at full slice resolution.
